[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/examples/diagnostic_robustness.ipynb)

# MISDA — diagnostic robustness to observation noise

This notebook studies how recovery changes as the observation-noise intensity `sigma` increases. It uses a focused subset of controlled diagnostics rather than repeating the whole routine benchmark. For each problem and replicate, clean `Z` and one standardized noise realization `epsilon` are generated once and reused across all sigma values, so only noise intensity changes along each curve. No sigma value is interpreted as a methodological cutoff.

In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test the local code. In Colab, install main.
target = ".[benchmarks]" if Path("pyproject.toml").exists() else "git+https://github.com/monacofj/misda.git@main#egg=misda[benchmarks]"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import misda
import misda.benchmarks as bench

N = 300
MISDA_SEED = 123
PROBLEM_IDS = (
    "independence",
    "total_redundancy",
    "blocks_4x5",
    "monotonic_redundancy",
    "antagonistic_linear_groups",
    "nonlinear_blocks_4x5",
    "antagonistic_nonlinear_groups",
    "transitive_chain",
)
SIGMAS = (0.00, 0.05, 0.10, 0.20, 0.40)
REPLICATE_SEEDS = (101, 202, 303, 404, 505)

## Controlled sweep design

The selected problems retain the original controls for independence, block redundancy, nonlinear block structure, signed conflict, and transitive chaining, and now also include the cases highlighted by the single-noise Pareto diagnostics: total redundancy, nonlinear monotonic redundancy, and antagonistic nonlinear groups. Each replicate seed determines an independent sample seed and observation-noise stream. MISDA's own seed is held fixed so the curves reflect sampled-data and observation variability rather than changes in the estimator's Monte Carlo stream.

The sigma grid samples the degradation curve around the `0.10` reference condition used by `benchmark_noisy.ipynb`; it does **not** define a pass/fail threshold.

In [ ]:
robustness_records = []

for problem_position, problem_id in enumerate(PROBLEM_IDS):
    problem = bench.PROBLEM_BY_ID[problem_id]
    for replicate_seed in REPLICATE_SEEDS:
        sample_seed = int(np.random.SeedSequence([replicate_seed, problem_position, 1]).generate_state(1)[0])
        observation_sequence = np.random.SeedSequence([replicate_seed, problem_position, 2])
        X = problem.sample(N=N, seed=sample_seed)
        Z = problem.evaluate(X)
        truth = bench.diagnostic_truth(problem, Z)
        epsilon = np.random.default_rng(observation_sequence).normal(size=Z.shape)

        for sigma in SIGMAS:
            Y = problem.observe(Z, sigma=sigma, standard_noise=epsilon)
            mis_set = misda.discover(Y, name=truth["name"], seed=MISDA_SEED)
            misda.evaluate(mis_set, metrics=("pareto",), candidates=1)
            benchmark_result = misda.benchmark(mis_set, truth)
            selected = mis_set.structural_ranking.selected
            selected_index = mis_set.structural_ranking.indices[0] if mis_set.structural_ranking.indices else None
            pareto_stability = mis_set.pareto_stability
            support_reasons = {
                reason
                for candidate_support in mis_set.support.results
                for reason in candidate_support.reasons
            }

            robustness_records.append({
                "problem_id": problem_id,
                "replicate_seed": replicate_seed,
                "sample_seed": sample_seed,
                "sigma": float(sigma),
                "latent_exact": bool(benchmark_result.latent_exact),
                "structural_exact": bool(benchmark_result.structural_dimension_exact),
                "selected_unit_adequacy": benchmark_result.assessment["selected_unit_adequacy"],
                "pareto_jaccard": benchmark_result.pareto_jaccard,
                "pareto_observation_jaccard": benchmark_result.observation_pareto_jaccard,
                "pareto_reduction_jaccard": selected.pareto.jaccard if selected is not None and selected.pareto is not None else np.nan,
                "pareto_end_to_end_jaccard": benchmark_result.pareto_jaccard,
                "pareto_observed_fraction": pareto_stability.observed_front_fraction,
                "pareto_additive_epsilon": pareto_stability.epsilon_for_candidate(selected_index) if selected_index is not None else np.nan,
                "pareto_dominance_margin_min": pareto_stability.dominance_margin_min,
                "pareto_dominance_margin_median": pareto_stability.dominance_margin_median,
                "pareto_dominance_margin_max": pareto_stability.dominance_margin_max,
                "support_status": mis_set.support.status,
                "supported": mis_set.support.status == "SUPPORTED",
                "unsupported": mis_set.support.status == "UNSUPPORTED",
                "transitive_chaining": "TRANSITIVE_CHAINING" in support_reasons,
            })

robustness = pd.DataFrame.from_records(robustness_records)

In [ ]:
def _mean_if_declared(series):
    values = series.dropna()
    return float(values.astype(float).mean()) if len(values) else np.nan

robustness_summary = (
    robustness.groupby(["problem_id", "sigma"], sort=False)
    .agg(
        replicates=("replicate_seed", "size"),
        latent_recovery=("latent_exact", "mean"),
        structural_recovery=("structural_exact", "mean"),
        selected_unit_recovery=("selected_unit_adequacy", _mean_if_declared),
        pareto_jaccard=("pareto_jaccard", "mean"),
        pareto_observation_jaccard=("pareto_observation_jaccard", "mean"),
        pareto_reduction_jaccard=("pareto_reduction_jaccard", "mean"),
        pareto_end_to_end_jaccard=("pareto_end_to_end_jaccard", "mean"),
        pareto_observed_fraction=("pareto_observed_fraction", "mean"),
        pareto_additive_epsilon=("pareto_additive_epsilon", "mean"),
        pareto_dominance_margin_min=("pareto_dominance_margin_min", "mean"),
        pareto_dominance_margin_median=("pareto_dominance_margin_median", "mean"),
        pareto_dominance_margin_max=("pareto_dominance_margin_max", "mean"),
        supported_rate=("supported", "mean"),
        unsupported_rate=("unsupported", "mean"),
        transitive_chaining_rate=("transitive_chaining", "mean"),
    )
    .reset_index()
)

print(robustness_summary.to_string(index=False))

## Reading the curves

`latent_recovery`, `structural_recovery`, and `selected_unit_recovery` are finite-replicate exact-recovery proportions, not fitted probabilities. The three Pareto Jaccards are deliberately separated: `pareto_observation_jaccard` compares observed `P_Y` with clean sampled truth `P_Z`; `pareto_reduction_jaccard` compares reduced `P_R` with observed `P_Y`; and `pareto_end_to_end_jaccard` (also retained as the legacy `pareto_jaccard`) compares `P_R` with `P_Z`. `pareto_observed_fraction`, range-normalized additive `epsilon+`, and the dominance-margin summaries use only observed `Y`; they characterize front saturation, full-space geometric approximation, and perturbation sensitivity without declaring a noise level or pass/fail cutoff. `supported_rate` and `unsupported_rate` remain dimensional-support diagnostics. For `transitive_chain`, dimensional recovery is expected to be poor already at `sigma=0`; the important robustness signal is whether the `TRANSITIVE_CHAINING` diagnostic itself persists.

In [ ]:
for metric in (
    "latent_recovery",
    "structural_recovery",
    "selected_unit_recovery",
    "pareto_observation_jaccard",
    "pareto_reduction_jaccard",
    "pareto_end_to_end_jaccard",
    "pareto_observed_fraction",
    "pareto_additive_epsilon",
    "pareto_dominance_margin_median",
    "supported_rate",
    "transitive_chaining_rate",
):
    pivot = robustness_summary.pivot(index="sigma", columns="problem_id", values=metric)
    ax = pivot.plot(marker="o", title=metric.replace("_", " ").title())
    ax.set_xlabel("sigma")
    ax.set_ylabel(metric)
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.25)
    plt.show()